In [2]:
class GameState:
    def __init__(self, routes, scores):
        self.routes = routes
        self.scores = scores

def copy_state(state):
    new_routes = []
    for r in state.routes:
        new_routes.append({"from": r["from"],"to": r["to"],"points": r["points"],"claimed": r["claimed"],"owner": r["owner"]})

    new_scores = {"MAX": state.scores["MAX"],"MIN": state.scores["MIN"]}

    return GameState(new_routes, new_scores)

def generate_moves(state):
    moves = []
    for route in state.routes:
        if route["claimed"] == False:
            moves.append(route)
    return moves

def apply_move(state, move, player):
    new_state = copy_state(state)

    for route in new_state.routes:
        if route["from"] == move["from"] and route["to"] == move["to"] and route["claimed"] == False:
            route["claimed"] = True
            route["owner"] = player
            new_state.scores[player] = new_state.scores[player] + route["points"]
            break

    return new_state

def connection_bonus(state, player):
    count = {}
    bonus = 0

    for route in state.routes:
        if route["owner"] == player:
            a = route["from"]
            b = route["to"]

            if a not in count:
                count[a] = 0
            if b not in count:
                count[b] = 0

            count[a] = count[a] + 1
            count[b] = count[b] + 1

    for city in count:
        if count[city] >= 2:
            bonus = bonus + 2

    return bonus

def block_penalty(state):
    penalty = 0
    for route in state.routes:
        if route["owner"] == "MIN" and route["points"] >= 5:
            penalty = penalty + 1
    return penalty

def evaluate(state):
    max_score = state.scores["MAX"]
    min_score = state.scores["MIN"]

    max_bonus = connection_bonus(state, "MAX")
    min_bonus = connection_bonus(state, "MIN")
    penalty = block_penalty(state)

    return (max_score + max_bonus) - (min_score + min_bonus) - penalty

def is_terminal(state):
    for route in state.routes:
        if route["claimed"] == False:
            return False
    return True

def minimax(state, depth, is_max_turn):
    if depth == 0 or is_terminal(state):
        return evaluate(state)

    moves = generate_moves(state)

    if is_max_turn:
        best = -9999
        for move in moves:
            new_state = apply_move(state, move, "MAX")
            value = minimax(new_state, depth - 1, False)
            if value > best:
                best = value
        return best
    else:
        best = 9999
        for move in moves:
            new_state = apply_move(state, move, "MIN")
            value = minimax(new_state, depth - 1, True)
            if value < best:
                best = value
        return best

def find_best_move(state, depth):
    moves = generate_moves(state)
    best_move = None
    best_value = -9999

    for move in moves:
        new_state = apply_move(state, move, "MAX")
        value = minimax(new_state, depth - 1, False)

        print("Move:", move["from"] + "-" + move["to"], "Points:", move["points"], "Value:", value)

        if value > best_value:
            best_value = value
            best_move = move

    return best_move, best_value

routes = [
    {"from": "A", "to": "B", "points": 5, "claimed": False, "owner": None},
    {"from": "A", "to": "C", "points": 3, "claimed": False, "owner": None},
    {"from": "A", "to": "D", "points": 4, "claimed": False, "owner": None},
    {"from": "B", "to": "C", "points": 2, "claimed": False, "owner": None},
    {"from": "B", "to": "D", "points": 6, "claimed": False, "owner": None},
    {"from": "C", "to": "D", "points": 2, "claimed": False, "owner": None},
    {"from": "C", "to": "E", "points": 5, "claimed": False, "owner": None},
    {"from": "D", "to": "E", "points": 6, "claimed": False, "owner": None}
]

scores = {"MAX": 0, "MIN": 0}
state = GameState(routes, scores)

best_move, best_value = find_best_move(state, 3)

print("\nBest move for MAX:")
print(best_move["from"] + "-" + best_move["to"])
print("Utility:", best_value)

Move: A-B Points: 5 Value: 4
Move: A-C Points: 3 Value: 3
Move: A-D Points: 4 Value: 5
Move: B-C Points: 2 Value: 2
Move: B-D Points: 6 Value: 6
Move: C-D Points: 2 Value: 3
Move: C-E Points: 5 Value: 4
Move: D-E Points: 6 Value: 6

Best move for MAX:
B-D
Utility: 6
